# Этап 0. Сбор данных: история коэффициентов АПЛ

Источник - [OddsPapi API](https://oddspapi.io) (`https://api.oddspapi.io/v4`).

**Результат:** `data/raw/epl_odds_raw.csv`. Это ответы API, развёрнутые в таблицу без изменений: одна строка - одна запись истории коэффициента.

**Запускать не обязательно:** готовая выгрузка уже лежит в репозитории. Ноутбук нужен, чтобы понимать, откуда взялись данные, и чтобы при необходимости сделать новую выгрузку (другие даты, букмекеры, рынки).

**Перед запуском:** в корне репозитория нужен файл `.env` со строкой `ODDSPAPI_KEY=ваш_ключ`

## Шаг 1. Библиотеки и ключ

In [7]:
import os
import time
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("ODDSPAPI_KEY")
print("Ключ найден:", API_KEY is not None)

Ключ найден: True


## Шаг 2. Параметры выгрузки

Чтобы выгрузить другой период, нужно изменить даты. API отдаёт матчи окном не больше 10 дней.

In [8]:
BASE = "https://api.oddspapi.io/v4"

DATE_FROM = "2026-09-11"
DATE_TO = "2026-09-20"

# Не больше 3 букмекеров в одном запросе, поэтому сразу делим на группы
BOOKMAKER_GROUPS = [["pinnacle", "singbet", "sbobet"], ["williamhill", "bet365"]]

# Рынки: 101 = исход матча 1X2, 1010 = тотал 2.5
MARKETS = ["101", "1010"]

RAW_CSV = Path("../data/raw/epl_odds_raw.csv")

## Шаг 3. Проверяем `/account`

In [ ]:
r = requests.get(f"{BASE}/account", params={"apiKey": API_KEY}, timeout=30)
print("Статус:", r.status_code)

Статус: 200


## Шаг 4. ID футбола: `/sports` 

ID уже узнал (футбол = 10), шаг можно пропустить. Он показывает, откуда взялось число.

In [10]:
r = requests.get(f"{BASE}/sports", params={"apiKey": API_KEY}, timeout=30)
print("Статус:", r.status_code)

sports = pd.DataFrame(r.json())
sports[sports.isin(["Soccer"]).any(axis=1)]

Статус: 200


,sportId,slug,sportName
0,10,soccer,Soccer


In [11]:
SPORT_ID = 10

## Шаг 5. ID Английской Премьер-лиги: `/tournaments` 

ID уже известен (АПЛ = 17), шаг можно пропустить. Фильтруем по точному совпадению страны и названия: в Англии есть и другие «Premier League» (молодёжная, женская и многое другое...)

In [ ]:
r = requests.get(f"{BASE}/tournaments", params={"apiKey": API_KEY, "sportId": SPORT_ID}, timeout=30)
print("Статус:", r.status_code)

tournaments = pd.DataFrame(r.json())
tournaments[(tournaments["categoryName"] == "England") & (tournaments["tournamentName"] == "Premier League")]

In [12]:
EPL_ID = 17

## Шаг 6. Матчи за период: `/fixtures`

API возвращает матчи всех футбольных лиг за период, АПЛ отбираем по `tournamentId`.

In [13]:
r = requests.get(f"{BASE}/fixtures",
                 params={"apiKey": API_KEY, "sportId": SPORT_ID, "from": DATE_FROM, "to": DATE_TO},
                 timeout=60)
print("Статус:", r.status_code)

fixtures = pd.DataFrame(r.json())
print("Всех футбольных матчей:", len(fixtures))

Статус: 200
Всех футбольных матчей: 8124


In [14]:
fixtures.head(5)

,fixtureId,participant1Id,participant2Id,sportId,tournamentId,seasonId,statusId,hasOdds,startTime,trueStartTime,...,participant1ShortName,participant1Abbr,participant2Name,participant2ShortName,participant2Abbr,sportName,tournamentSlug,categorySlug,categoryName,tournamentName
0,id1001924274321096,266593,25353,10,19242,143890.0,2.0,False,2026-09-11T00:00:00.000Z,2026-09-10T23:59:06.000Z,...,Managua FC,MAN,Real Esteli FC,Esteli,REA,Soccer,primera-division,nicaragua,Nicaragua,Primera Division
1,id1002617673353370,456823,460711,10,26176,144976.0,2.0,False,2026-09-11T00:00:00.000Z,None,...,Creighton Bluejays,CRE,Saint Marys Gaels,Saint Marys Gaels,SAI,Soccer,ncaa-regular-season,usa,USA,"NCAA, Regular Season"
2,id1002617673353372,460509,460681,10,26176,144976.0,2.0,False,2026-09-11T00:00:00.000Z,None,...,UIC Flames,UIC,Western Illinois Leathernecks,Western Illinois Leathernecks,WES,Soccer,ncaa-regular-season,usa,USA,"NCAA, Regular Season"
3,id1002617673353374,460673,460529,10,26176,144976.0,2.0,False,2026-09-11T00:00:00.000Z,None,...,Omaha Mavericks,OMA,Yale Bulldogs,Yale Bulldogs,YAL,Soccer,ncaa-regular-season,usa,USA,"NCAA, Regular Season"
4,id1002617873250594,475220,475244,10,26178,144930.0,2.0,False,2026-09-11T00:00:00.000Z,None,...,Texas Southern Lady Tigers,TEX,Stephen F. Austin Lumberjacks,Stephen F. Austin,SFA,Soccer,ncaa-regular-season-women,usa,USA,"NCAA Regular Season, Women"


In [16]:
epl = fixtures[fixtures["tournamentId"] == EPL_ID].sort_values("fixtureId")
print("Матчей АПЛ:", len(epl))
epl[["fixtureId", "participant1Name", "participant2Name"]]

Матчей АПЛ: 16


,fixtureId,participant1Name,participant2Name
1485,id1000001772221252,Aston Villa,Nottingham Forest
1486,id1000001772221254,AFC Bournemouth,Brentford FC
1487,id1000001772221256,Chelsea FC,Hull City
3208,id1000001772221258,Coventry City,Brighton & Hove Albion
1488,id1000001772221260,Crystal Palace,Ipswich Town
4388,id1000001772221262,Leeds United,Newcastle United
1489,id1000001772221264,Liverpool FC,Fulham FC
3679,id1000001772221266,Manchester United,Manchester City
2505,id1000001772221268,Sunderland AFC,Arsenal FC
2249,id1000001772221270,Tottenham Hotspur,Everton FC


## Шаг 7. История коэффициентов: `/historical-odds` (если что квоту не тратит)

По каждому матчу два запроса, по одному на группу букмекеров. Между запросами API требует паузу 5 секунд: 16 матчей примерно 3 минуты.

Ответ вложенный: букмекер - рынок - исход - игрок - список записей (здест есть пример ответа https://oddspapi.io/ru/docs/get-historical-odds). Для рынков 1X2 и тотала «игрок» всегда `"0"`, поле нужно только для ставок на игроков. Ответ большой (в нём все рынки букмекера), поэтому сразу разворачиваем его и берём только рынки из `MARKETS`.

Каждая запись — словарь `createdAt`, `price`, `limit`, `active`, `exchangeMeta`. Кладём её в строку как есть и добавляем, к какому матчу, букмекеру, рынку и исходу она относится.

*Сделаю оговорку, что для выполнения данного шага пришлось прибегнуть к использованию ИИ 

In [17]:
rows = []

for fixture_id in epl["fixtureId"]:
    for group in BOOKMAKER_GROUPS:
        r = requests.get(f"{BASE}/historical-odds",
                         params={"apiKey": API_KEY, "fixtureId": fixture_id, "bookmakers": ",".join(group)},
                         timeout=60)
        time.sleep(5.5)

        for bookmaker, bookmaker_data in r.json()["bookmakers"].items():
            for market_id, market_data in bookmaker_data["markets"].items():
                if market_id not in MARKETS:
                    continue
                for outcome_id, outcome_data in market_data["outcomes"].items():
                    for player_id, history in outcome_data["players"].items():
                        for record in history:
                            rows.append({"fixtureId": fixture_id, "bookmaker": bookmaker,
                                         "marketId": market_id, "outcomeId": outcome_id,
                                         "playerId": player_id, **record})

    print("Готово:", fixture_id, "| строк:", len(rows))

Готово: id1000001772221252 | строк: 4967
Готово: id1000001772221254 | строк: 9348
Готово: id1000001772221256 | строк: 13370
Готово: id1000001772221258 | строк: 17390
Готово: id1000001772221260 | строк: 21927
Готово: id1000001772221262 | строк: 25981
Готово: id1000001772221264 | строк: 30362
Готово: id1000001772221266 | строк: 35153
Готово: id1000001772221268 | строк: 39521
Готово: id1000001772221270 | строк: 44836
Готово: id1000001772221274 | строк: 49662
Готово: id1000001772221276 | строк: 54095
Готово: id1000001772221278 | строк: 57798
Готово: id1000001772221286 | строк: 60999
Готово: id1000001772221288 | строк: 65455
Готово: id1000001772221290 | строк: 71003


## Шаг 8. Собираем таблицу

In [18]:
odds = pd.DataFrame(rows)
print("Строк:", len(odds))
odds.head()

Строк: 71003


,fixtureId,bookmaker,marketId,outcomeId,playerId,createdAt,price,limit,active,exchangeMeta
0,id1000001772221252,pinnacle,101,101,0,2026-08-30T19:48:20.892Z,1.781,320.0,True,None
1,id1000001772221252,pinnacle,101,101,0,2026-08-30T19:48:26.392Z,1.806,310.0,True,None
2,id1000001772221252,pinnacle,101,101,0,2026-08-30T19:48:46.525Z,1.813,307.0,True,None
3,id1000001772221252,pinnacle,101,101,0,2026-08-30T19:48:46.829Z,1.909,275.0,True,None
4,id1000001772221252,pinnacle,101,101,0,2026-08-30T20:46:01.164Z,1.934,267.0,True,None


## Шаг 9. Добавляем время начала и команды из `/fixtures`

В ответе `/historical-odds` есть только ID матча. Берём из `/fixtures` три поля: `startTime`, `participant1Name` (хозяева), `participant2Name` (гости).

In [19]:
match_info = epl[["fixtureId", "startTime", "participant1Name", "participant2Name"]]
odds = match_info.merge(odds, on="fixtureId")
odds.head()

,fixtureId,startTime,participant1Name,participant2Name,bookmaker,marketId,outcomeId,playerId,createdAt,price,limit,active,exchangeMeta
0,id1000001772221252,2026-09-12T14:00:00.000Z,Aston Villa,Nottingham Forest,pinnacle,101,101,0,2026-08-30T19:48:20.892Z,1.781,320.0,True,None
1,id1000001772221252,2026-09-12T14:00:00.000Z,Aston Villa,Nottingham Forest,pinnacle,101,101,0,2026-08-30T19:48:26.392Z,1.806,310.0,True,None
2,id1000001772221252,2026-09-12T14:00:00.000Z,Aston Villa,Nottingham Forest,pinnacle,101,101,0,2026-08-30T19:48:46.525Z,1.813,307.0,True,None
3,id1000001772221252,2026-09-12T14:00:00.000Z,Aston Villa,Nottingham Forest,pinnacle,101,101,0,2026-08-30T19:48:46.829Z,1.909,275.0,True,None
4,id1000001772221252,2026-09-12T14:00:00.000Z,Aston Villa,Nottingham Forest,pinnacle,101,101,0,2026-08-30T20:46:01.164Z,1.934,267.0,True,None


## Шаг 10. Сохраняем

In [20]:
# про RAW_CSV написано в шаге 2 если что 

RAW_CSV.parent.mkdir(parents=True, exist_ok=True)
odds.to_csv(RAW_CSV, index=False)
print("Сохранено:", RAW_CSV, odds.shape)

Сохранено: ../data/raw/epl_odds_raw.csv (71003, 13)
